# 05 · Governed retrieval: tiers, escalation and citations

Tier the data by safety, escalate when the first search is not confident, and check every citation.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`04_ner_grounding_rerank.ipynb`](04_ner_grounding_rerank.ipynb) / [`06_monitoring_and_comparison.ipynb`](06_monitoring_and_comparison.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

### Recap from earlier notebooks

Governed retrieval builds on two pieces you have already seen: the BM25 query builder (`bm25_query`, notebook 02) and the vector refinement step (`vector_refine`, notebook 04), plus the system prompt from notebook 04. They are repeated here, unchanged, so this notebook runs on its own.

In [ ]:
def bm25_query(text, k, entities=None, tier=None, exclude=None, require=None, recipe_ids=None):
    """BM25 over name and text (`source` and `entities` are exact-match fields), with optional hard filters and an entity boost."""
    filters, must_not, should = [], [], []
    if recipe_ids:
        filters.append({"terms": {"recipe_id": list(recipe_ids)}})
    if tier:
        filters.append({"term": {"tier": tier}})
    for field, values in (exclude or {}).items():
        must_not.append({"terms": {field: [v.lower() for v in values]}})
    for field, values in (require or {}).items():
        filters += [{"term": {field: v.lower()}} for v in values]
    if entities:
        should.append({"constant_score": {"filter": {"terms": {"entities": entities}}, "boost": 6.0}})
    must = [{"multi_match": {"query": text, "fields": ["recipe_name^4", "entities^3", "text^2", "source"],
                             "type": "best_fields", "operator": "or"}}]
    bool_query = {"must": must, "filter": filters, "must_not": must_not}
    if should:
        bool_query["should"] = should
    return {"size": k, "_source": RECIPE_FIELDS + ["entities"], "query": {"bool": bool_query}}


NER_RERANK_SYSTEM = (VECTOR_SYSTEM + " Never recommend a recipe that the context marks as containing an excluded allergen.")


def vector_refine(question, recipe_ids, top_k=5, exclude=None, pool=300):
    """kNN restricted to the grounding candidates. `pool` must cover the whole corpus, because the ID filter
    is applied after the nearest-neighbour search."""
    filters = [{"terms": {"recipe_id": list(recipe_ids)}}]
    must_not = [{"terms": {f: [v.lower() for v in vs]}} for f, vs in (exclude or {}).items()]
    body = {"size": top_k, "_source": RECIPE_FIELDS,
            "query": {"bool": {"must": [{"knn": {"embedding": {"vector": embed(question), "k": max(pool, len(recipe_ids))}}}],
                               "filter": filters, "must_not": must_not}}}
    return normalize_hits(search(IDX["vector"], body))

## Governance tiering

**The problem.** Searching everything equally treats safety-critical data like any other, and leaves no record of *what was searched or why*.

**The fix.** Tier the data by safety, then search in order:

- **`hot` tier:** recipes with any allergen caution. Searched first.
- **`lt` (long-term) tier:** recipes without cautions. Searched only if the hot search is not confident, which is called **escalation**.

Escalate if any of these is true: no hot candidates, fewer than 3 candidates, a top BM25 score below 15, or the question **names a source website** that is missing from the hot results.

**Citations.** Each retrieved recipe gets a handle (`[R1]`, `[R2]`, ...). The model must cite a handle after each factual sentence, and a check confirms it did not cite a handle that was never retrieved.

**Auditing.** The journey writes one audit record per query. The account used here is read-only, so this notebook builds the record and shows it, but **does not write it**.

*Mirrors:* `scripts/governance-tiering/src/query.py`.

In [ ]:
MIN_HOT_SCORE, MIN_HOT_CANDIDATES, REFINE_ANN_POOL = 15.0, 3, 300
CITATION_RE = re.compile(r"\[(R\d+)\]")
DOMAIN_RE = re.compile(r"\b(?=[\w-]*[A-Za-z])[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}\b")


def governed_retrieve(question, top_k=5, candidate_k=25, exclude=None, require=None):
    entities = extract_entities(question)
    hot = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities, tier="hot",
                                                        exclude=exclude, require=require)))
    top_score = hot[0]["score"] if hot else 0.0
    named = [m.group(0).lower() for m in DOMAIN_RE.finditer(question)]
    hot_sources = {(h.get("source") or "").lower() for h in hot}
    named_missing = bool(named) and not any(tok in src for tok in named for src in hot_sources)

    escalate = (not hot or len(hot) < MIN_HOT_CANDIDATES or top_score < MIN_HOT_SCORE or named_missing)
    reason = None
    if escalate:
        reason = ("no_hot_candidates" if not hot else "named_source_not_in_hot_tier" if named_missing
                  else "low_confidence_hot_score")
        grounding = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities,
                                                                  exclude=exclude, require=require)))
        tiers = ["hot", "lt"]
    else:
        grounding, tiers = hot, ["hot"]

    ids = [h["recipe_id"] for h in grounding if h.get("recipe_id")]
    refined = vector_refine(question, ids, top_k, exclude, pool=REFINE_ANN_POOL) if ids else []
    refined = [{**h, "handle": f"R{i}"} for i, h in enumerate(refined, start=1)]
    meta = {"tiers_searched": tiers, "escalated": escalate, "escalation_reason": reason,
            "hot_hit_count": len(hot), "hot_top_score": round(top_score, 3),
            "evidence_handles": [h["handle"] for h in refined]}
    return grounding, refined, meta, entities


def governed_context(hits):
    return "\n\n".join(
        f"[{h['handle']}]\nrecipe_name={h.get('recipe_name')}, source={h.get('source')}, tier={h.get('tier')}, "
        f"cautions={', '.join(h.get('cautions_display') or []) or 'none listed'}\n{(h.get('text') or '')[:900]}\n[/{h['handle']}]"
        for h in hits) or "No recipes matched the requested constraints."


def governed_system(handles):
    if not handles:
        return NER_RERANK_SYSTEM + " No recipe evidence was retrieved, so say you don't know rather than inventing a citation."
    allowed = " ".join(f"[{h}]" for h in handles)
    return (NER_RERANK_SYSTEM + f" CITATION RULES (mandatory): each recipe snippet is wrapped in a handle tag like [R1]. "
            f"Allowed citation tags: {allowed}. After every sentence with a factual recipe claim, append the [R#] tag(s) "
            "that support it. Use opening tags only (e.g. [R1]). Never invent a tag that is not in the allowed list.")


def ask_governed(question, top_k=5):
    t0 = time.time()
    grounding, refined, meta, entities = governed_retrieve(question, top_k)
    handles = meta["evidence_handles"]
    reply = answer(f"Question:\n{question}\n\nContext:\n{governed_context(refined)}", governed_system(handles))
    cited = list(dict.fromkeys(CITATION_RE.findall(reply)))
    bad = [h for h in cited if h not in handles]
    meta.update(cited_handles=cited, hallucinated_citations=bad, citations_valid=not bad)
    audit_record = {"question": question, "entities": entities, **meta, "answer": reply,
                    "latency_ms": round((time.time() - t0) * 1000, 1)}
    return reply, grounding, refined, meta, audit_record


for question in [Q_WOMENS, Q_GRATEFUL]:
    reply, grounding, refined, meta, audit_record = ask_governed(question)
    domain = "thegratefulgirlcooks" if "grateful" in question else "womensweeklyfood"
    print("\nQ:", question)
    print(f"  GOVERNANCE: tiers_searched={meta['tiers_searched']} escalated={meta['escalated']} "
          f"reason={meta['escalation_reason']} hot_hits={meta['hot_hit_count']} hot_top_score={meta['hot_top_score']}")
    show_hits(refined, limit=5)
    print(f"  -> the named recipe ({domain}) is in the evidence: {has_source(refined, domain)}")
    print(f"  CITATIONS: evidence={meta['evidence_handles']} cited={meta['cited_handles']} "
          f"hallucinated={meta['hallucinated_citations']}")
    print("  ANSWER:", reply)

**Expected:**

- **`Q_WOMENS`** names a recipe that carries a Sulfites caution, so it lives in the `hot` tier. The hot search finds it, so there is **no escalation** (`tiers_searched=['hot']`).
- **`Q_GRATEFUL`** names a recipe with no cautions, so it lives in the `lt` tier. The hot search cannot find it, so the system **escalates**: `tiers_searched=['hot', 'lt']`, `reason=named_source_not_in_hot_tier`, and the right recipe now appears.
- With `RUN_LLM = True`, the answer cites `[R1]`-style handles, and `hallucinated=[]` means every cited handle was really retrieved. With it off, the citation lines are empty.

The escalation decision does not need an LLM, so it works either way. Here is the record the journey would have written to the audit log (it is not written from this notebook):

In [ ]:
print(json.dumps({k: audit_record[k] for k in
                  ["question", "entities", "tiers_searched", "escalated", "escalation_reason",
                   "evidence_handles", "cited_handles", "citations_valid", "latency_ms"]}, indent=2))

**Reading the record.** This is the record for the last question in the loop above (the `thegratefulgirlcooks.com` one). With `RUN_LLM = False`, `citations_valid: true` only means that nothing was cited. It answers, after the fact, "did we search the archive tier, and why?" and "which retrieved chunk backs each claim?". Notebook 06 reads these records back to watch for problems.

**Try this:** ask `ask_governed("Is there a recipe with no allergen cautions?")` and see whether it escalates.

### Checkpoint

You should be able to say why the question about the `thegratefulgirlcooks.com` recipe escalates to the long-term tier (the recipe has no cautions, so it is not in the hot tier) and the question about the `womensweeklyfood.com.au` recipe does not.

Next: open [`06_monitoring_and_comparison.ipynb`](06_monitoring_and_comparison.ipynb).